# Spark et Databricks (Partie 3) : colonnes, lignes et tris

📖 **Le mini-cours complet** : [Spark et Databricks : maîtriser colonnes, lignes et tris en PySpark (Partie 3)](https://www.monshotdata.com/p/spark-databricks-colonnes-lignes-tris-partie-3)

🧭 **Le cours** : [Bonnes pratiques d'ingénierie › 1.4 Passage à l'échelle et calcul distribué](https://www.monshotdata.com/mini-cours/bonnes-pratiques-ingenierie#h2sc14)

---

Une fois ton DataFrame chargé, tout le travail consiste à le transformer : choisir et créer des colonnes, corriger les types, supprimer les doublons, filtrer les lignes et trier. Ici tu passes en revue toutes ces opérations en PySpark.

**À la fin de ce notebook, tu sauras :**

- référencer, sélectionner, créer et renommer des colonnes (`select()`, `withColumn()`, `lit()`, `alias()`) ;
- convertir les types avec `cast()`, supprimer colonnes et doublons ;
- combiner, découper et conditionner des colonnes (`concat()`, `split()`, `when()`) ;
- filtrer les lignes, y compris sur le texte et les valeurs nulles, puis trier avec `orderBy()`.

### Avant de lancer

Ce notebook tourne sur **Databricks**, pas sur Colab. Un compte **Databricks Free Edition** (gratuit) suffit.

- **Ouvrir le notebook** : dans Databricks, Workspace → Import, puis dépose ce fichier `.ipynb` (ou colle l'URL du fichier sur GitHub).
- **Le lancer** : attache-le au calcul **serverless**, puis exécute les cellules de haut en bas (ou « Run all »).
- **Les données** : la première cellule recrée le Volume `workspace.default.cours_spark` et le fichier `employes.csv` de la Partie 1 : tu peux lancer ce notebook sans avoir fait les parties précédentes.
- **Les erreurs voulues** : quelques cellules provoquent l'erreur expliquée dans le mini-cours. Elle est capturée par un `try / except` et affichée, pour que l'exécution aille au bout.

🧪 *Testé le 21 novembre 2026 : exécution complète de haut en bas sur Databricks Free Edition (calcul serverless, Spark 4.2, Python 3.12). Les sorties affichées viennent de cette exécution ; les identifiants techniques (adresse de la session, noms de fichiers `part-…`, `uid` des modèles) changent à chaque lancement.*

## 1. Préparation

Ce notebook s'exécute seul : cette cellule recrée le Volume et le fichier `employes.csv` de la Partie 1 (même code).

In [1]:
import numpy as np
import pandas as pd

# Crée un Volume : un espace de stockage de fichiers dans Unity Catalog
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.cours_spark")
chemin_volume = "/Volumes/workspace/default/cours_spark"

# Dataset fictif de 1 000 employés (graine fixe : résultats reproductibles)
rng = np.random.default_rng(42)
n = 1000
prenoms = ["Léa", "Hugo", "Emma", "Louis", "Chloé",
           "Lucas", "Inès", "Nathan", "Jade", "Adam"]
noms = ["Martin", "Bernard", "Dubois", "Thomas", "Robert",
        "Richard", "Petit", "Durand", "Leroy", "Moreau"]
villes = ["Paris", "Lyon", "Marseille", "Toulouse", "Bordeaux", "Lille"]
postes = ["Data Analyst", "Data Scientist", "Data Engineer",
          "ML Engineer", "Manager"]

employes = pd.DataFrame({
    "id_employe": np.arange(1001, 1001 + n),
    "prenom": rng.choice(prenoms, n),
    "nom": rng.choice(noms, n),
    "age": rng.integers(22, 61, n),
    "ville": rng.choice(villes, n),
    "poste": rng.choice(postes, n),
    "salaire": rng.integers(32, 95, n) * 1000,
})

# Écrit le fichier CSV dans le Volume
employes.to_csv(f"{chemin_volume}/employes.csv", index=False)
print(employes.shape)

# Résultat :
# (1000, 7)

(1000, 7)


## 2. Les objets Column

Un DataFrame est aussi une collection d'objets `Column`, que tu sélectionnes avec `select()`.

In [2]:
donnees = [
    ("1003", "Pierre", "Morel", 22, "20000", 5.3),
    ("2903", "Julie", "Granier", 26, "40000", 9.2),
    ("1231", "Marc", "Joly", 25, "35000", 3.5),
    ("2122", "Katia", "Chen", 29, "65000", 7.3),
]
schema = ("id_emp STRING, prenom STRING, nom STRING, age INT, "
          "salaire STRING, credit FLOAT")
df = spark.createDataFrame(donnees, schema=schema)
df.show()

# Trois façons d'obtenir un objet Column
print(df["prenom"])
print(df.prenom)
print(df[1])   # par position

# +------+------+-------+---+-------+------+
# |id_emp|prenom|    nom|age|salaire|credit|
# +------+------+-------+---+-------+------+
# |  1003|Pierre|  Morel| 22|  20000|   5.3|
# |  2903| Julie|Granier| 26|  40000|   9.2|
# |  1231|  Marc|   Joly| 25|  35000|   3.5|
# |  2122| Katia|   Chen| 29|  65000|   7.3|
# +------+------+-------+---+-------+------+
# Column<'prenom'>
# Column<'prenom'>
# Column<'prenom'>

+------+------+-------+---+-------+------+
|id_emp|prenom|    nom|age|salaire|credit|
+------+------+-------+---+-------+------+
|  1003|Pierre|  Morel| 22|  20000|   5.3|
|  2903| Julie|Granier| 26|  40000|   9.2|
|  1231|  Marc|   Joly| 25|  35000|   3.5|
|  2122| Katia|   Chen| 29|  65000|   7.3|
+------+------+-------+---+-------+------+

Column<'prenom'>
Column<'prenom'>
Column<'prenom'>


In [3]:
df.select("prenom").show()                  # un nom de colonne
df.select(df["age"]).show()                 # un objet Column
df.select(["prenom", "nom"]).show()         # une liste de noms
df.select(["prenom", df["age"]]).show()     # noms et objets mélangés
df.select("prenom", df[3]).show()           # sans liste : arguments séparés

# Résultat des deux dernières lignes (identique) :
# +------+---+
# |prenom|age|
# +------+---+
# |Pierre| 22|
# | Julie| 26|
# |  Marc| 25|
# | Katia| 29|
# +------+---+

+------+
|prenom|
+------+
|Pierre|
| Julie|
|  Marc|
| Katia|
+------+

+---+
|age|
+---+
| 22|
| 26|
| 25|
| 29|
+---+

+------+-------+
|prenom|    nom|
+------+-------+
|Pierre|  Morel|
| Julie|Granier|
|  Marc|   Joly|
| Katia|   Chen|
+------+-------+

+------+---+
|prenom|age|
+------+---+
|Pierre| 22|
| Julie| 26|
|  Marc| 25|
| Katia| 29|
+------+---+

+------+---+
|prenom|age|
+------+---+
|Pierre| 22|
| Julie| 26|
|  Marc| 25|
| Katia| 29|
+------+---+



## 3. Transformer et ajouter des colonnes

Expressions de colonnes, `withColumn()`, `lit()`, comparaisons et renommage.

In [4]:
from pyspark.sql.functions import expr

# Une expression sur une colonne crée une nouvelle colonne
df.select(df["salaire"] * 1.1).show()
# +---------------+
# |(salaire * 1.1)|
# +---------------+
# |        22000.0|
# |        44000.0|
# |        38500.0|
# |        71500.0|
# +---------------+

# expr() transforme une chaîne SQL en colonne
df.select(expr("CAST(salaire AS INT)")).printSchema()
# root
#  |-- salaire: integer (nullable = true)

# selectExpr() : la même chose, sans importer expr()
df.selectExpr("CAST(salaire AS INT) AS salaire_int", "credit AS bonus").show()
# +-----------+-----+
# |salaire_int|bonus|
# +-----------+-----+
# |      20000|  5.3|
# |      40000|  9.2|
# |      35000|  3.5|
# |      65000|  7.3|
# +-----------+-----+

+---------------+
|(salaire * 1.1)|
+---------------+
|        22000.0|
|        44000.0|
|        38500.0|
|        71500.0|
+---------------+

root
 |-- salaire: integer (nullable = true)

+-----------+-----+
|salaire_int|bonus|
+-----------+-----+
|      20000|  5.3|
|      40000|  9.2|
|      35000|  3.5|
|      65000|  7.3|
+-----------+-----+



In [5]:
# Ajouter une colonne : withColumn() renvoie un nouveau DataFrame
df.withColumn("nouveau_salaire", df["salaire"] * 1.1).show()

# Plusieurs colonnes d'un coup : withColumns() et un dictionnaire
df.withColumns({
    "salaire_x2": df["salaire"] * 2,
    "credit_x2": df["credit"] * 2,
}).show()

# +------+------+-------+---+-------+------+----------+------------------+
# |id_emp|prenom|    nom|age|salaire|credit|salaire_x2|         credit_x2|
# +------+------+-------+---+-------+------+----------+------------------+
# |  1003|Pierre|  Morel| 22|  20000|   5.3|     40000|10.600000381469727|
# |  2903| Julie|Granier| 26|  40000|   9.2|     80000|18.399999618530273|
# |  1231|  Marc|   Joly| 25|  35000|   3.5|     70000|               7.0|
# |  2122| Katia|   Chen| 29|  65000|   7.3|    130000|14.600000381469727|
# +------+------+-------+---+-------+------+----------+------------------+

+------+------+-------+---+-------+------+---------------+
|id_emp|prenom|    nom|age|salaire|credit|nouveau_salaire|
+------+------+-------+---+-------+------+---------------+
|  1003|Pierre|  Morel| 22|  20000|   5.3|        22000.0|
|  2903| Julie|Granier| 26|  40000|   9.2|        44000.0|
|  1231|  Marc|   Joly| 25|  35000|   3.5|        38500.0|
|  2122| Katia|   Chen| 29|  65000|   7.3|        71500.0|
+------+------+-------+---+-------+------+---------------+

+------+------+-------+---+-------+------+----------+------------------+
|id_emp|prenom|    nom|age|salaire|credit|salaire_x2|         credit_x2|
+------+------+-------+---+-------+------+----------+------------------+
|  1003|Pierre|  Morel| 22|  20000|   5.3|     40000|10.600000381469727|
|  2903| Julie|Granier| 26|  40000|   9.2|     80000|18.399999618530273|
|  1231|  Marc|   Joly| 25|  35000|   3.5|     70000|               7.0|
|  2122| Katia|   Chen| 29|  65000|   7.3|    130000|14.600000381469727|
+------+------+-

In [6]:
from pyspark.sql.functions import lit

df.withColumn("constante", lit(10)).show()
df.withColumn("texte", lit("Bonjour")).withColumn("drapeau", lit(True)).show()

# +------+------+-------+---+-------+------+-------+-------+
# |id_emp|prenom|    nom|age|salaire|credit|  texte|drapeau|
# +------+------+-------+---+-------+------+-------+-------+
# |  1003|Pierre|  Morel| 22|  20000|   5.3|Bonjour|   true|
# |  2903| Julie|Granier| 26|  40000|   9.2|Bonjour|   true|
# |  1231|  Marc|   Joly| 25|  35000|   3.5|Bonjour|   true|
# |  2122| Katia|   Chen| 29|  65000|   7.3|Bonjour|   true|
# +------+------+-------+---+-------+------+-------+-------+

+------+------+-------+---+-------+------+---------+
|id_emp|prenom|    nom|age|salaire|credit|constante|
+------+------+-------+---+-------+------+---------+
|  1003|Pierre|  Morel| 22|  20000|   5.3|       10|
|  2903| Julie|Granier| 26|  40000|   9.2|       10|
|  1231|  Marc|   Joly| 25|  35000|   3.5|       10|
|  2122| Katia|   Chen| 29|  65000|   7.3|       10|
+------+------+-------+---+-------+------+---------+

+------+------+-------+---+-------+------+-------+-------+
|id_emp|prenom|    nom|age|salaire|credit|  texte|drapeau|
+------+------+-------+---+-------+------+-------+-------+
|  1003|Pierre|  Morel| 22|  20000|   5.3|Bonjour|   true|
|  2903| Julie|Granier| 26|  40000|   9.2|Bonjour|   true|
|  1231|  Marc|   Joly| 25|  35000|   3.5|Bonjour|   true|
|  2122| Katia|   Chen| 29|  65000|   7.3|Bonjour|   true|
+------+------+-------+---+-------+------+-------+-------+



In [7]:
df.select(df["credit"] > 5).show()
df.select(df["prenom"] == "Pierre").show()
df.select((df["credit"] > 5) & (df["salaire"] > 25000)).show()
df.select(df["salaire"].between(30000, 70000)).show()

# +------------+  +-----------------+  +------------------------------------+
# |(credit > 5)|  |(prenom = Pierre)|  |((credit > 5) AND (salaire > 25000))|
# +------------+  +-----------------+  +------------------------------------+
# |        true|  |             true|  |                               false|
# |        true|  |            false|  |                                true|
# |       false|  |            false|  |                               false|
# |        true|  |            false|  |                                true|
# +------------+  +-----------------+  +------------------------------------+
#
# between(30000, 70000) : false, true, true, true

+------------+
|(credit > 5)|
+------------+
|        true|
|        true|
|       false|
|        true|
+------------+

+-----------------+
|(prenom = Pierre)|
+-----------------+
|             true|
|            false|
|            false|
|            false|
+-----------------+

+------------------------------------+
|((credit > 5) AND (salaire > 25000))|
+------------------------------------+
|                               false|
|                                true|
|                               false|
|                                true|
+------------------------------------+

+-------------------------------------------+
|((salaire >= 30000) AND (salaire <= 70000))|
+-------------------------------------------+
|                                      false|
|                                       true|
|                                       true|
|                                       true|
+-------------------------------------------+



In [8]:
# Sans alias, le nom de la colonne reprend l'opération
print(df.select(df["salaire"] * 1.1).columns)
# ['(salaire * 1.1)']

# alias() donne un nom lisible
df.select((df["salaire"] * 1.1).alias("nouveau_salaire")).show()
# +---------------+
# |nouveau_salaire|
# +---------------+
# |        22000.0|
# |        44000.0|
# |        38500.0|
# |        71500.0|
# +---------------+

# Renommer une colonne, puis plusieurs
print(df.withColumnRenamed("id_emp", "id_employe").columns)
print(df.withColumnsRenamed({"prenom": "first_name", "nom": "last_name"}).columns)
# ['id_employe', 'prenom', 'nom', 'age', 'salaire', 'credit']
# ['id_emp', 'first_name', 'last_name', 'age', 'salaire', 'credit']

['(salaire * 1.1)']
+---------------+
|nouveau_salaire|
+---------------+
|        22000.0|
|        44000.0|
|        38500.0|
|        71500.0|
+---------------+

['id_employe', 'prenom', 'nom', 'age', 'salaire', 'credit']
['id_emp', 'first_name', 'last_name', 'age', 'salaire', 'credit']


## 4. Types, suppressions et doublons

`cast()`, `drop()`, `dropDuplicates()` et `distinct()`.

In [9]:
# Changer le type : cast() puis réaffectation à la même colonne
df_int = df.withColumn("salaire", df["salaire"].cast("int"))
df_int.printSchema()
# root
#  |-- id_emp: string (nullable = true)
#  |-- prenom: string (nullable = true)
#  |-- nom: string (nullable = true)
#  |-- age: integer (nullable = true)
#  |-- salaire: integer (nullable = true)
#  |-- credit: float (nullable = true)

# Conversion automatique : texte x nombre fonctionne... mais à éviter
df.select(df["salaire"] * (1 + df["credit"] / 100)).show()
# +--------------------------------+
# |(salaire * (1 + (credit / 100)))|
# +--------------------------------+
# |              21060.000038146973|
# |              43679.999923706055|
# |                         36225.0|
# |               69745.00012397766|
# +--------------------------------+

root
 |-- id_emp: string (nullable = true)
 |-- prenom: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salaire: integer (nullable = true)
 |-- credit: float (nullable = true)

+--------------------------------+
|(salaire * (1 + (credit / 100)))|
+--------------------------------+
|              21060.000038146973|
|              43679.999923706055|
|                         36225.0|
|               69745.00012397766|
+--------------------------------+



In [10]:
df.drop("credit", "age").show()
# +------+------+-------+-------+
# |id_emp|prenom|    nom|salaire|
# +------+------+-------+-------+
# |  1003|Pierre|  Morel|  20000|
# |  2903| Julie|Granier|  40000|
# |  1231|  Marc|   Joly|  35000|
# |  2122| Katia|   Chen|  65000|
# +------+------+-------+-------+

# Supprimer une colonne qui n'existe pas ne lève pas d'erreur
print(df.select("*").drop("colonne_inexistante").columns)
# ['id_emp', 'prenom', 'nom', 'age', 'salaire', 'credit']

+------+------+-------+-------+
|id_emp|prenom|    nom|salaire|
+------+------+-------+-------+
|  1003|Pierre|  Morel|  20000|
|  2903| Julie|Granier|  40000|
|  1231|  Marc|   Joly|  35000|
|  2122| Katia|   Chen|  65000|
+------+------+-------+-------+

['id_emp', 'prenom', 'nom', 'age', 'salaire', 'credit']


In [11]:
donnees_dup = [
    ("1003", "Pierre", "Morel", 22, "20000", 5.3),
    ("2903", "Julie", "Granier", 26, "40000", 9.2),
    ("1231", "Marc", "Joly", 25, "35000", 3.5),
    ("2903", "Julie", "Granier", 26, "40000", 9.2),   # doublon exact
    ("2122", "Katia", "Chen", 29, "65000", 7.3),
]
df_dup = spark.createDataFrame(donnees_dup, schema=schema)
print(df_dup.count(), "lignes")          # 5 lignes
df_dup.dropDuplicates().show()           # 4 lignes : le doublon a disparu

# Doublons définis sur un sous-ensemble de colonnes
donnees_ex = [
    ("1003", "Pierre", "Morel", 22, "20000", 5.3),
    ("2903", "Julie", "Granier", 28, "45000", 8.9),
    ("1231", "Marc", "Joly", 25, "35000", 3.5),
    ("2903", "Julie", "Granier", 26, "40000", 9.2),
    ("2122", "Katia", "Chen", 29, "65000", 7.3),
]
df_ex = spark.createDataFrame(donnees_ex, schema=schema)
print(df_ex.dropDuplicates().count(),
      df_ex.dropDuplicates(subset=["id_emp", "prenom", "nom"]).count())
# 5 4

5 lignes
+------+------+-------+---+-------+------+
|id_emp|prenom|    nom|age|salaire|credit|
+------+------+-------+---+-------+------+
|  1003|Pierre|  Morel| 22|  20000|   5.3|
|  2903| Julie|Granier| 26|  40000|   9.2|
|  1231|  Marc|   Joly| 25|  35000|   3.5|
|  2122| Katia|   Chen| 29|  65000|   7.3|
+------+------+-------+---+-------+------+

5 4


In [12]:
df_dup.select("id_emp").distinct().show()
df_dup.select("prenom", "nom").distinct().show()
# +------+   +------+-------+
# |id_emp|   |prenom|    nom|
# +------+   +------+-------+
# |  1003|   |Pierre|  Morel|
# |  2903|   | Julie|Granier|
# |  1231|   |  Marc|   Joly|
# |  2122|   | Katia|   Chen|
# +------+   +------+-------+

# En Pandas, unique() marche sur une Series, pas sur un DataFrame
pdf = df_dup.toPandas()
print(pdf["id_emp"].unique())
# ['1003' '2903' '1231' '2122']
try:
    pdf[["prenom", "nom"]].unique()
except Exception as erreur:
    print(type(erreur).__name__, ":", erreur)
# AttributeError: 'DataFrame' object has no attribute 'unique'

+------+
|id_emp|
+------+
|  1003|
|  2903|
|  1231|
|  2122|
+------+

+------+-------+
|prenom|    nom|
+------+-------+
|Pierre|  Morel|
| Julie|Granier|
|  Marc|   Joly|
| Katia|   Chen|
+------+-------+

['1003' '2903' '1231' '2122']
AttributeError : 'DataFrame' object has no attribute 'unique'


## 5. Combiner, découper et conditionner

`concat()`, `split()`, `when()` et `describe()`.

In [13]:
from pyspark.sql.functions import concat, concat_ws

df.select(concat("prenom", "nom").alias("nom_complet")).show()
df.select(concat("prenom", lit(" "), "nom").alias("nom_complet")).show()
df.select(concat_ws(" ", "prenom", "nom").alias("nom_complet")).show()

# concat() seul :     avec lit(" ") ou concat_ws() :
# +------------+      +-------------+
# | nom_complet|      |  nom_complet|
# +------------+      +-------------+
# | PierreMorel|      | Pierre Morel|
# |JulieGranier|      |Julie Granier|
# |    MarcJoly|      |    Marc Joly|
# |   KatiaChen|      |   Katia Chen|
# +------------+      +-------------+

+------------+
| nom_complet|
+------------+
| PierreMorel|
|JulieGranier|
|    MarcJoly|
|   KatiaChen|
+------------+

+-------------+
|  nom_complet|
+-------------+
| Pierre Morel|
|Julie Granier|
|    Marc Joly|
|   Katia Chen|
+-------------+

+-------------+
|  nom_complet|
+-------------+
| Pierre Morel|
|Julie Granier|
|    Marc Joly|
|   Katia Chen|
+-------------+



In [14]:
from pyspark.sql.functions import split, col

df_nc = df.select(concat_ws(" ", "prenom", "nom").alias("nom_complet"))
df_nc.withColumn("morceaux", split("nom_complet", " ")).show()
# +-------------+----------------+
# |  nom_complet|        morceaux|
# +-------------+----------------+
# | Pierre Morel| [Pierre, Morel]|
# |Julie Granier|[Julie, Granier]|
# |    Marc Joly|    [Marc, Joly]|
# |   Katia Chen|   [Katia, Chen]|
# +-------------+----------------+

(
    df_nc.withColumn("morceaux", split("nom_complet", " "))
    .withColumn("prenom", col("morceaux")[0])
    .withColumn("nom", col("morceaux")[1])
    .drop("morceaux")
).show()
# +-------------+------+-------+
# |  nom_complet|prenom|    nom|
# +-------------+------+-------+
# | Pierre Morel|Pierre|  Morel|
# |Julie Granier| Julie|Granier|
# |    Marc Joly|  Marc|   Joly|
# |   Katia Chen| Katia|   Chen|
# +-------------+------+-------+

# La même chose avec select(), sans garder les autres colonnes
(
    df_nc.select(split("nom_complet", " ").alias("morceaux"))
    .select(col("morceaux")[0].alias("prenom"), col("morceaux")[1].alias("nom"))
).show()

+-------------+----------------+
|  nom_complet|        morceaux|
+-------------+----------------+
| Pierre Morel| [Pierre, Morel]|
|Julie Granier|[Julie, Granier]|
|    Marc Joly|    [Marc, Joly]|
|   Katia Chen|   [Katia, Chen]|
+-------------+----------------+

+-------------+------+-------+
|  nom_complet|prenom|    nom|
+-------------+------+-------+
| Pierre Morel|Pierre|  Morel|
|Julie Granier| Julie|Granier|
|    Marc Joly|  Marc|   Joly|
|   Katia Chen| Katia|   Chen|
+-------------+------+-------+

+------+-------+
|prenom|    nom|
+------+-------+
|Pierre|  Morel|
| Julie|Granier|
|  Marc|   Joly|
| Katia|   Chen|
+------+-------+



In [15]:
from pyspark.sql.functions import when, col

# 1. when() seul : NULL si la condition est fausse
df.select("prenom", "age", when(col("age") > 25, "Senior")
                           .alias("categorie")).show()
# 2. Deux when() enchaînés
df.select("prenom", "age", when(col("age") > 25, "Senior")
                           .when(col("age") > 24, "Confirmé")
                           .alias("categorie")).show()
# 3. Les mêmes, dans l'autre ordre
df.select("prenom", "age", when(col("age") > 24, "Confirmé")
                           .when(col("age") > 25, "Senior")
                           .alias("categorie")).show()
# 4. otherwise() pour toutes les autres lignes
df.select("prenom", "age", when(col("age") > 25, "Senior")
                           .otherwise("Junior").alias("categorie")).show()

# prenom  age |    1.    |    2.    |    3.    |   4.
# Pierre   22 |   NULL   |   NULL   |   NULL   | Junior
# Julie    26 |  Senior  |  Senior  | Confirmé | Senior
# Marc     25 |   NULL   | Confirmé | Confirmé | Junior
# Katia    29 |  Senior  |  Senior  | Confirmé | Senior

+------+---+---------+
|prenom|age|categorie|
+------+---+---------+
|Pierre| 22|     NULL|
| Julie| 26|   Senior|
|  Marc| 25|     NULL|
| Katia| 29|   Senior|
+------+---+---------+

+------+---+---------+
|prenom|age|categorie|
+------+---+---------+
|Pierre| 22|     NULL|
| Julie| 26|   Senior|
|  Marc| 25| Confirmé|
| Katia| 29|   Senior|
+------+---+---------+

+------+---+---------+
|prenom|age|categorie|
+------+---+---------+
|Pierre| 22|     NULL|
| Julie| 26| Confirmé|
|  Marc| 25| Confirmé|
| Katia| 29| Confirmé|
+------+---+---------+

+------+---+---------+
|prenom|age|categorie|
+------+---+---------+
|Pierre| 22|   Junior|
| Julie| 26|   Senior|
|  Marc| 25|   Junior|
| Katia| 29|   Senior|
+------+---+---------+



In [16]:
chemin_volume = "/Volumes/workspace/default/cours_spark"
df_employes = spark.read.csv(f"{chemin_volume}/employes.csv",
                             header=True, inferSchema=True)
df_employes.describe(["age", "salaire"]).show()

# +-------+------------------+------------------+
# |summary|               age|           salaire|
# +-------+------------------+------------------+
# |  count|              1000|              1000|
# |   mean|             41.15|           63025.0|
# | stddev|11.267584410975527|17860.105216889355|
# |    min|                22|             32000|
# |    max|                60|             94000|
# +-------+------------------+------------------+
# (df_employes.describe() sans argument
# couvre aussi les colonnes texte :
#  min et max seulement)

+-------+------------------+------------------+
|summary|               age|           salaire|
+-------+------------------+------------------+
|  count|              1000|              1000|
|   mean|             41.15|           63025.0|
| stddev|11.267584410975527|17860.105216889355|
|    min|                22|             32000|
|    max|                60|             94000|
+-------+------------------+------------------+



## 6. Filtrer les lignes

`filter()`, filtres sur le texte et sur les valeurs nulles.

In [17]:
print(type(df_int["prenom"]))   # <class 'pyspark.sql.connect.column.Column'>
print(type(df_int.head()))      # <class 'pyspark.sql.types.Row'>

df_int.filter(df_int["salaire"] > 40000).show()
df_int.filter((df_int["age"] > 25) & (df_int["salaire"] > 40000)).show()
df_int.filter("age > 25 AND salaire > 40000").show()
# Les trois renvoient la même ligne :
# +------+------+----+---+-------+------+
# |id_emp|prenom| nom|age|salaire|credit|
# +------+------+----+---+-------+------+
# |  2122| Katia|Chen| 29|  65000|   7.3|
# +------+------+----+---+-------+------+

df_int.filter(df_int["salaire"].between(40000, 70000)).show()    # Julie et Katia
df_int.filter(df_int["prenom"].isin(["Pierre", "Julie"])).show()  # Pierre et Julie

<class 'pyspark.sql.connect.column.Column'>
<class 'pyspark.sql.types.Row'>
+------+------+----+---+-------+------+
|id_emp|prenom| nom|age|salaire|credit|
+------+------+----+---+-------+------+
|  2122| Katia|Chen| 29|  65000|   7.3|
+------+------+----+---+-------+------+

+------+------+----+---+-------+------+
|id_emp|prenom| nom|age|salaire|credit|
+------+------+----+---+-------+------+
|  2122| Katia|Chen| 29|  65000|   7.3|
+------+------+----+---+-------+------+

+------+------+----+---+-------+------+
|id_emp|prenom| nom|age|salaire|credit|
+------+------+----+---+-------+------+
|  2122| Katia|Chen| 29|  65000|   7.3|
+------+------+----+---+-------+------+

+------+------+-------+---+-------+------+
|id_emp|prenom|    nom|age|salaire|credit|
+------+------+-------+---+-------+------+
|  2903| Julie|Granier| 26|  40000|   9.2|
|  2122| Katia|   Chen| 29|  65000|   7.3|
+------+------+-------+---+-------+------+

+------+------+-------+---+-------+------+
|id_emp|prenom|    

In [18]:
from pyspark.sql.functions import length, col

df.filter(col("prenom").startswith("J")).show()   # Julie
df.filter(col("nom").endswith("er")).show()       # Granier
df.filter(col("nom").like("%er%")).show()         # Granier
df.filter(col("prenom").rlike("^M.*")).show()     # Marc
df.filter(length(col("prenom")) > 5).show()       # Pierre

# Exemple de sortie (rlike) :
# +------+------+----+---+-------+------+
# |id_emp|prenom| nom|age|salaire|credit|
# +------+------+----+---+-------+------+
# |  1231|  Marc|Joly| 25|  35000|   3.5|
# +------+------+----+---+-------+------+

+------+------+-------+---+-------+------+
|id_emp|prenom|    nom|age|salaire|credit|
+------+------+-------+---+-------+------+
|  2903| Julie|Granier| 26|  40000|   9.2|
+------+------+-------+---+-------+------+

+------+------+-------+---+-------+------+
|id_emp|prenom|    nom|age|salaire|credit|
+------+------+-------+---+-------+------+
|  2903| Julie|Granier| 26|  40000|   9.2|
+------+------+-------+---+-------+------+

+------+------+-------+---+-------+------+
|id_emp|prenom|    nom|age|salaire|credit|
+------+------+-------+---+-------+------+
|  2903| Julie|Granier| 26|  40000|   9.2|
+------+------+-------+---+-------+------+

+------+------+----+---+-------+------+
|id_emp|prenom| nom|age|salaire|credit|
+------+------+----+---+-------+------+
|  1231|  Marc|Joly| 25|  35000|   3.5|
+------+------+----+---+-------+------+

+------+------+-----+---+-------+------+
|id_emp|prenom|  nom|age|salaire|credit|
+------+------+-----+---+-------+------+
|  1003|Pierre|Morel| 22|  2

In [19]:
donnees_nulles = [
    ("1003", "Pierre", None, 22, "20000", 5.3),
    ("2903", "Julie", "Granier", 26, None, 9.2),
    ("1231", "Marc", None, 25, "35000", None),
    ("2122", "Katia", "Chen", 29, "65000", 7.3),
]
df_null = spark.createDataFrame(donnees_nulles, schema=schema)
df_null.filter(col("nom").isNull()).show()
df_null.filter(col("nom").isNotNull()).show()
df_null.filter("nom IS NULL").show()   # même résultat que isNull()

# isNull() :
# +------+------+----+---+-------+------+
# |id_emp|prenom| nom|age|salaire|credit|
# +------+------+----+---+-------+------+
# |  1003|Pierre|NULL| 22|  20000|   5.3|
# |  1231|  Marc|NULL| 25|  35000|  NULL|
# +------+------+----+---+-------+------+
# isNotNull() : Julie et Katia

+------+------+----+---+-------+------+
|id_emp|prenom| nom|age|salaire|credit|
+------+------+----+---+-------+------+
|  1003|Pierre|NULL| 22|  20000|   5.3|
|  1231|  Marc|NULL| 25|  35000|  NULL|
+------+------+----+---+-------+------+

+------+------+-------+---+-------+------+
|id_emp|prenom|    nom|age|salaire|credit|
+------+------+-------+---+-------+------+
|  2903| Julie|Granier| 26|   NULL|   9.2|
|  2122| Katia|   Chen| 29|  65000|   7.3|
+------+------+-------+---+-------+------+

+------+------+----+---+-------+------+
|id_emp|prenom| nom|age|salaire|credit|
+------+------+----+---+-------+------+
|  1003|Pierre|NULL| 22|  20000|   5.3|
|  1231|  Marc|NULL| 25|  35000|  NULL|
+------+------+----+---+-------+------+



## 7. Trier un DataFrame

`orderBy()`, croissant, décroissant, sur plusieurs colonnes ou sur une expression.

In [20]:
from pyspark.sql.functions import desc, col

df_int.orderBy("age").show()                      # Pierre, Marc, Julie, Katia
df_int.orderBy(desc("age")).show()                # Katia, Julie, Marc, Pierre
df_int.orderBy(desc("age"), "salaire").show()     # âge décroissant, puis salaire
df_int.orderBy(col("salaire") * (1 + col("credit"))).show()

# Tri sur l'expression salaire x (1 + crédit) :
# +------+------+-------+---+-------+------+
# |id_emp|prenom|    nom|age|salaire|credit|
# +------+------+-------+---+-------+------+
# |  1003|Pierre|  Morel| 22|  20000|   5.3|
# |  1231|  Marc|   Joly| 25|  35000|   3.5|
# |  2903| Julie|Granier| 26|  40000|   9.2|
# |  2122| Katia|   Chen| 29|  65000|   7.3|
# +------+------+-------+---+-------+------+

+------+------+-------+---+-------+------+
|id_emp|prenom|    nom|age|salaire|credit|
+------+------+-------+---+-------+------+
|  1003|Pierre|  Morel| 22|  20000|   5.3|
|  1231|  Marc|   Joly| 25|  35000|   3.5|
|  2903| Julie|Granier| 26|  40000|   9.2|
|  2122| Katia|   Chen| 29|  65000|   7.3|
+------+------+-------+---+-------+------+

+------+------+-------+---+-------+------+
|id_emp|prenom|    nom|age|salaire|credit|
+------+------+-------+---+-------+------+
|  2122| Katia|   Chen| 29|  65000|   7.3|
|  2903| Julie|Granier| 26|  40000|   9.2|
|  1231|  Marc|   Joly| 25|  35000|   3.5|
|  1003|Pierre|  Morel| 22|  20000|   5.3|
+------+------+-------+---+-------+------+

+------+------+-------+---+-------+------+
|id_emp|prenom|    nom|age|salaire|credit|
+------+------+-------+---+-------+------+
|  2122| Katia|   Chen| 29|  65000|   7.3|
|  2903| Julie|Granier| 26|  40000|   9.2|
|  1231|  Marc|   Joly| 25|  35000|   3.5|
|  1003|Pierre|  Morel| 22|  20000|   5.3|
+------+-

## 8. Ce qu'il faut retenir

- Pour référencer une colonne, préfère `df["col"]` ou `col("col")` à `df.col`.
- `withColumn()`, `withColumns()`, `alias()` et `withColumnsRenamed()` renvoient tous un nouveau DataFrame : affecte-le ou enchaîne.
- Convertis les types explicitement avec `cast()` plutôt que de compter sur la conversion automatique.
- `when()` s'évalue comme un if/elif : l'ordre des conditions compte, et `otherwise()` évite les NULL.
- Filtre avec `filter()` ou une chaîne SQL, en entourant chaque condition de parenthèses, et trie avec `orderBy()` et `desc()`.

---

## Pour aller plus loin

- [Spark et Databricks : créer, lire et écrire des DataFrames PySpark (Partie 2)](https://www.monshotdata.com/p/spark-databricks-dataframes-pyspark-partie-2) — la partie précédente : créer, lire, écrire et afficher des DataFrames
- [Spark et Databricks : agrégations, jointures et fonctions de fenêtre en PySpark (Partie 4)](https://www.monshotdata.com/p/spark-databricks-agregations-jointures-fenetres-partie-4) — la partie suivante : groupBy, pivot, jointures, fonctions de fenêtre et valeurs manquantes